In [0]:
GRANT USE CATALOG ON CATALOG workspace TO
`9821ee15-08cc-4e13-8c4d-2e6d52afaaba`;
GRANT USE SCHEMA ON SCHEMA workspace.olist_gold TO
`9821ee15-08cc-4e13-8c4d-2e6d52afaaba`;
GRANT READ VOLUME ON VOLUME workspace.olist_gold.parquet TO
`9821ee15-08cc-4e13-8c4d-2e6d52afaaba`;

In [0]:
#Bibliotecas
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

# ==========================================
# CONFIGURAÇÕES
# ==========================================

SILVER_PATH = "/Volumes/workspace/olist_silver/parquet/"
GOLD_PATH = "/Volumes/workspace/olist_gold/parquet/"

# ==========================================
# LEITURA DAS TABELAS SILVER
# ==========================================

def read_silver(table_name):
    return spark.read.parquet(f"{SILVER_PATH}{table_name}/")

customers = read_silver("olist_customers_dataset")
geolocation = read_silver("olist_geolocation_dataset")
order_items = read_silver("olist_order_items_dataset")
payments = read_silver("olist_order_payments_dataset")
reviews = read_silver("olist_order_reviews_dataset")
orders = read_silver("olist_orders_dataset")
products = read_silver("olist_products_dataset")
sellers = read_silver("olist_sellers_dataset")
category_translation = read_silver("product_category_name_translation")

print("Todas as tabelas Silver foram carregadas!")

In [0]:
# ==========================================
# AGREGAÇÃO DE PAGAMENTOS POR PEDIDO
# ==========================================

payments_order = (
    payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("total_payment"),
        F.max("payment_installments").alias("max_installments")
    )
)

# ==========================================
# TRADUÇÃO DAS CATEGORIAS
# ==========================================

products_enriched = (
    products
    .join(
        category_translation,
        on="product_category_name",
        how="left"
    )
    .withColumn(
        "category",
        F.coalesce(
            F.col("product_category_name"),
            F.lit("categoria_nao_identificada")
        )
    )
    .withColumn(
        "category_english",
        F.coalesce(
            F.col("product_category_name_english"),
            F.lit("unknown")
        )
    )
)

# ==========================================
# BASE ANALÍTICA DE ITENS
# ==========================================

base = (
    order_items.alias("i")
    .join(orders.alias("o"), "order_id", "inner")
    .join(products_enriched.alias("p"), "product_id", "left")
    .join(sellers.alias("s"), "seller_id", "left")
    .join(customers.alias("c"), "customer_id", "left")
    .join(payments_order.alias("pay"), "order_id", "left")
    .select(
        F.col("i.order_id"),
        F.col("i.order_item_id"),
        F.col("i.product_id"),
        F.col("i.seller_id"),
        F.col("o.customer_id"),
        F.col("c.customer_unique_id"),
        F.col("s.seller_state").alias("uf_origem"),
        F.col("s.seller_zip_code_prefix").alias("cep_origem"),
        F.col("c.customer_zip_code_prefix").alias("cep_destino"),
        F.col("c.customer_state").alias("uf_destino"),
        F.col("i.price"),
        F.col("i.freight_value"),
        F.col("p.product_weight_g"),
        F.col("p.category"),
        F.col("p.category_english"),
        F.col("o.order_status"),
        F.col("o.order_purchase_timestamp"),
        F.col("o.order_delivered_customer_date"),
        F.col("o.order_estimated_delivery_date"),
        F.col("pay.total_payment"),
        F.col("pay.max_installments"),
    )
    .withColumn(
        "mes_pedido",
        F.date_trunc("month", "order_purchase_timestamp").cast("date")
    )
    .withColumn(
        "atraso",
        F.when(
            F.col("order_delivered_customer_date").isNotNull() &
            F.col("order_estimated_delivery_date").isNotNull() &
            (F.col("order_delivered_customer_date") >
             F.col("order_estimated_delivery_date")),
            1
        ).otherwise(0)
    )
)

# Pedidos válidos para indicadores comerciais.
# Cancelados e indisponíveis não entram no desempenho de mercado.

base_comercial = base.filter(
    ~F.col("order_status").isin("canceled", "unavailable")
)

print("Base analítica preparada!")
print("Registros:", base.count())

In [0]:
gold_agg_mercado = (
    base_comercial
    .groupBy("mes_pedido", "category")
    .agg(
        F.countDistinct("order_id").alias("total_pedidos"),
        F.countDistinct("customer_unique_id").alias("compradores_unicos"),
        F.countDistinct("seller_id").alias("vendedores_ativos"),
        F.sum("order_item_id").alias("itens_vendidos"),
        F.sum("price").cast(DecimalType(18, 2)).alias("volume_vendas"),
        F.avg("price").cast(DecimalType(18, 2)).alias("preco_medio")
    )
)

window_mercado = (
    Window
    .partitionBy("category")
    .orderBy("mes_pedido")
)

gold_agg_mercado = (
    gold_agg_mercado
    .withColumn(
        "pedidos_mes_anterior",
        F.lag("total_pedidos").over(window_mercado)
    )
    .withColumn(
        "crescimento_pct",
        F.when(
            F.col("pedidos_mes_anterior") > 0,
            F.round(
                (
                    F.col("total_pedidos") -
                    F.col("pedidos_mes_anterior")
                ) / F.col("pedidos_mes_anterior") * 100,
                2
            )
        )
    )
)

print("Gold Mercado construída!")

In [0]:
gold_agg_frete = (
    base_comercial
    .groupBy("category", "uf_origem")
    .agg(
        F.countDistinct("order_id").alias("total_pedidos"),
        F.avg("freight_value").alias("frete_medio"),
        F.avg("price").alias("preco_medio"),
        F.avg("product_weight_g").alias("peso_medio_g"),
        F.avg("max_installments").alias("parcelamento_medio"),
        F.sum("freight_value").cast(DecimalType(18, 2)).alias("frete_total"),
        F.sum("price").cast(DecimalType(18, 2)).alias("valor_total_produtos")
    )
    .withColumn(
        "frete_pct_preco",
        F.when(
            F.col("preco_medio") > 0,
            F.round(
                F.col("frete_medio") / F.col("preco_medio") * 100,
                2
            )
        )
    )
    .withColumn(
        "ticket_medio",
        F.round(
            F.col("valor_total_produtos") /
            F.col("total_pedidos"),
            2
        )
    )
    .withColumn("frete_medio", F.round("frete_medio", 2))
    .withColumn("preco_medio", F.round("preco_medio", 2))
    .withColumn("peso_medio_g", F.round("peso_medio_g", 2))
    .withColumn("parcelamento_medio", F.round("parcelamento_medio", 2))
)

print("Gold Frete construída!")

In [0]:
# ==========================================
# GEOLOCALIZAÇÃO POR CEP
# ==========================================

geo = (
    geolocation
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.avg("geolocation_lat").alias("latitude"),
        F.avg("geolocation_lng").alias("longitude")
    )
)

geo_origem = geo.select(
    F.col("geolocation_zip_code_prefix").alias("cep_origem"),
    F.col("latitude").alias("lat_origem"),
    F.col("longitude").alias("lon_origem")
)

geo_destino = geo.select(
    F.col("geolocation_zip_code_prefix").alias("cep_destino"),
    F.col("latitude").alias("lat_destino"),
    F.col("longitude").alias("lon_destino")
)

# ==========================================
# BASE DE ROTAS
# ==========================================

rotas = (
    base_comercial
    .filter(
        F.col("order_delivered_customer_date").isNotNull() &
        F.col("uf_origem").isNotNull() &
        F.col("uf_destino").isNotNull()
    )
    .select(
        "order_id",
        "category",
        "uf_origem",
        "uf_destino",
        "cep_origem",
        "cep_destino",
        "atraso"
    )
    .dropDuplicates()
    .join(geo_origem, "cep_origem", "left")
    .join(geo_destino, "cep_destino", "left")
)

# ==========================================
# DISTÂNCIA HAVERSINE (KM)
# ==========================================

lat1 = F.radians(F.col("lat_origem"))
lat2 = F.radians(F.col("lat_destino"))
lon1 = F.radians(F.col("lon_origem"))
lon2 = F.radians(F.col("lon_destino"))

a = (
    F.pow(F.sin((lat2 - lat1) / 2), 2) +
    F.cos(lat1) * F.cos(lat2) *
    F.pow(F.sin((lon2 - lon1) / 2), 2)
)

rotas = rotas.withColumn(
    "distancia_km",
    F.when(
        F.col("lat_origem").isNotNull() &
        F.col("lat_destino").isNotNull(),
        6371 * 2 * F.asin(
            F.sqrt(F.least(F.lit(1.0), a))
        )
    )
)

# ==========================================
# AGREGAÇÃO POR ROTA
# ==========================================

gold_ref_rotas = (
    rotas
    .groupBy("category", "uf_origem", "uf_destino")
    .agg(
        F.countDistinct("order_id").alias("total_pedidos"),
        F.avg("distancia_km").alias("distancia_media_km"),
        F.sum("atraso").alias("pedidos_atrasados")
    )
    .withColumn(
        "risco_atraso_pct",
        F.round(
            F.col("pedidos_atrasados") /
            F.col("total_pedidos") * 100,
            2
        )
    )
    .withColumn(
        "distancia_media_km",
        F.round("distancia_media_km", 0)
    )
)

print("Gold Rotas construída!")

In [0]:
# ==========================================
# AVALIAÇÕES POR PEDIDO
# ==========================================

reviews_order = (
    reviews
    .filter(F.col("review_score").isNotNull())
    .groupBy("order_id")
    .agg(
        F.avg("review_score").alias("nota_media_pedido"),
        F.countDistinct("review_id").alias("total_avaliacoes"),
        F.sum(
            F.when(F.col("review_score") <= 2, 1).otherwise(0)
        ).alias("avaliacoes_baixas")
    )
)

# ==========================================
# CATEGORIAS PRESENTES EM CADA PEDIDO
# ==========================================

order_categories = (
    base_comercial
    .select("order_id", "category")
    .distinct()
)

reviews_by_category = (
    order_categories
    .join(reviews_order, "order_id", "inner")
)

# ==========================================
# CLASSIFICAÇÃO SIMPLES DE RECLAMAÇÕES
# ==========================================

reviews_classified = (
    reviews
    .withColumn(
        "reclamacao",
        F.when(
            F.lower(F.coalesce(
                F.col("review_comment_message"), F.lit("")
            )).rlike("nao recebi|não recebi|nao chegou|não chegou"),
            "Não recebi"
        )
        .when(
            F.lower(F.coalesce(
                F.col("review_comment_message"), F.lit("")
            )).rlike("atras|demor"),
            "Atraso na entrega"
        )
        .when(
            F.lower(F.coalesce(
                F.col("review_comment_message"), F.lit("")
            )).rlike("defeit|danific|quebrad"),
            "Produto danificado"
        )
        .when(
            F.lower(F.coalesce(
                F.col("review_comment_message"), F.lit("")
            )).rlike("diferente|errado|incompleto"),
            "Produto diferente do esperado"
        )
        .otherwise("Outros / não identificado")
    )
)

complaints_by_category = (
    order_categories
    .join(
        reviews_classified.select("order_id", "reclamacao"),
        "order_id",
        "inner"
    )
    .groupBy("category", "reclamacao")
    .agg(F.count("*").alias("quantidade"))
)

window_complaints = (
    Window
    .partitionBy("category")
    .orderBy(F.desc("quantidade"), F.asc("reclamacao"))
)

most_common_complaint = (
    complaints_by_category
    .withColumn("ranking", F.row_number().over(window_complaints))
    .filter(F.col("ranking") == 1)
    .select(
        "category",
        F.col("reclamacao").alias("queixa_mais_comum")
    )
)

# ==========================================
# GOLD REPUTAÇÃO
# ==========================================

gold_agg_reputacao = (
    reviews_by_category
    .groupBy("category")
    .agg(
        F.avg("nota_media_pedido").alias("nota_media"),
        F.sum("total_avaliacoes").alias("total_avaliacoes"),
        F.sum("avaliacoes_baixas").alias("avaliacoes_baixas")
    )
    .withColumn(
        "notas_1_2_pct",
        F.round(
            F.col("avaliacoes_baixas") /
            F.col("total_avaliacoes") * 100,
            2
        )
    )
    .join(most_common_complaint, "category", "left")
    .withColumn("nota_media", F.round("nota_media", 2))
)

print("Gold Reputação construída!")

In [0]:
gold_agg_categoria = (
    base_comercial
    .groupBy("mes_pedido", "category", "category_english")
    .agg(
        F.countDistinct("order_id").alias("total_pedidos"),
        F.count("*").alias("total_itens"),
        F.countDistinct("seller_id").alias("vendedores_ativos"),
        F.sum("price").cast(DecimalType(18, 2)).alias("faturamento_produtos"),
        F.avg("price").alias("preco_medio"),
        F.min("price").alias("preco_minimo"),
        F.expr("percentile_approx(price, 0.5)").alias("preco_mediano"),
        F.max("price").alias("preco_maximo"),
        F.avg("product_weight_g").alias("peso_medio_g"),
        F.avg("freight_value").alias("frete_medio")
    )
    .withColumn(
        "frete_pct_preco",
        F.when(
            F.col("preco_medio") > 0,
            F.round(
                F.col("frete_medio") /
                F.col("preco_medio") * 100,
                2
            )
        )
    )
    .withColumn("preco_medio", F.round("preco_medio", 2))
    .withColumn("preco_mediano", F.round("preco_mediano", 2))
    .withColumn("peso_medio_g", F.round("peso_medio_g", 2))
    .withColumn("frete_medio", F.round("frete_medio", 2))
)

print("Gold Categoria construída!")

In [0]:
# ==========================================
# DICIONÁRIO DAS TABELAS GOLD
# ==========================================

gold_tables = {
    "gold_agg_mercado": gold_agg_mercado,
    "gold_agg_frete": gold_agg_frete,
    "gold_ref_rotas": gold_ref_rotas,
    "gold_agg_reputacao": gold_agg_reputacao,
    "gold_agg_categoria": gold_agg_categoria
}

# ==========================================
# GRAVAÇÃO NO VOLUME GOLD
# ==========================================

for table_name, dataframe in gold_tables.items():

    output_path = f"{GOLD_PATH}{table_name}/"

    (
        dataframe
        .write
        .mode("overwrite")
        .parquet(output_path)
    )

    print(f"Salvo com sucesso: {table_name}")

print("\nTodas as tabelas Gold foram gravadas!")

In [0]:
for table_name in gold_tables.keys():

    path = f"{GOLD_PATH}{table_name}/"

    df = spark.read.parquet(path)

    print(f"\n{'='*55}")
    print(f"TABELA: {table_name}")
    print(f"REGISTROS: {df.count()}")
    print("SCHEMA:")

    df.printSchema()

    display(df.limit(5))